In [ ]:
import sys
import introns
import importlib
import warnings
import pickle

import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt

from Bio import SeqIO
from Bio.SeqUtils import gc_fraction
from termcolor import colored
from sklearn.utils import class_weight
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, precision_recall_fscore_support, multilabel_confusion_matrix

importlib.reload(introns)

In [ ]:
#file = "./fastas_and_gff/introns_for_classifier_training.fasta"
file = "./fastas_and_gff/introns_for_3-class_classifier_training.fasta"
#file = "./fastas_and_gff/introns_for_4-class_classifier_training.fasta"

GCs = [gc_fraction(record.seq) for record in SeqIO.parse(file, "fasta")]
lengths = [len(record.seq) for record in SeqIO.parse(file, "fasta")]
mean_GC = sum(GCs)/len(GCs)
mean_length = sum(lengths)/len(lengths)
print(mean_GC, mean_length)

In [ ]:
teal_color = "#599191"

In [ ]:
classifiers_acc_prec_rec_f = {}

## Trenowanie
4 klasy: konw, niekonw, inne pozycje, losowe sekwencje

In [ ]:
### sprawdzanie ze introny nie sa swoimi wlasnymi wariantami

seqs = [i for i in SeqIO.parse("./fastas_and_gff/introns_for_4-class_classifier_training.fasta", 'fasta')]
for id1 in range(len(seqs)-1):
    s1 = seqs[id1]
    for id2 in range(id1+1, len(seqs)):
        s2 = seqs[id2]
        if s1.seq==s2.seq:
            print(s1.description)
            print(s2.description)
            print('\n')

In [ ]:
introns_seqs, introns_classes, introns_characteristics = [], [], []
f = open("./fastas_and_gff/introns_for_4-class_classifier_training.fasta", 'r')
for l in f.readlines():
    if l.startswith(">"):
        t = l.split()[-1]
        intron_type = 0 if t=="intron_K" else (1 if t=="intron_NK" else (2 if t=="intron_other" else 3))
    else:
        prev_exon_seq, intron_seq, next_exon_seq = l[:5], l[5:-5], l[-5:]
        its_characteristics = introns.compute_intron_characteristics(l)
        introns_seqs.append((prev_exon_seq, intron_seq, next_exon_seq))
        introns_classes.append(intron_type)
        introns_characteristics.append(its_characteristics)
print(len(introns_seqs), len(introns_classes), len(introns_characteristics))

X_train, X_test, y_train, y_test = train_test_split(introns_characteristics, introns_classes, test_size=0.3, random_state=420)
introns_classes = np.array(introns_classes).reshape(-1,1)
classes_weights_4head = class_weight.compute_sample_weight(class_weight='balanced', y=y_train)

In [ ]:
#sprawdzanie korelacji miedzy cechami
pd.DataFrame(introns_characteristics).corr()

In [8]:
#clf_4head = xgb.XGBClassifier(use_label_encoder=False, eval_metric='aucpr', objective='multi:softmax')
clf_4head = xgb.XGBClassifier(use_label_encoder=False,
                              eval_metric=balanced_accuracy_score,
                              objective='multi:softmax',
                              seed=123)
clf_4head.fit(X_train, y_train, sample_weight=classes_weights_4head);

pred_4head = clf_4head.predict(X_test)

KeyboardInterrupt: 

In [ ]:
acc = balanced_accuracy_score(y_test,pred_4head)
print("4-class classifier accuracy score:", acc)

precision, recall, fscore, _ = precision_recall_fscore_support(y_test,pred_4head)
print(f'''
Precision:    {precision}
recall:       {recall}
f-score:      {fscore}
''')
classifiers_acc_prec_rec_f['bac 4-class'] = (acc, precision, recall, fscore)
print("confusion matrix:", multilabel_confusion_matrix(y_test,pred_4head))

In [ ]:
### pickle.dump(clf_4head, open('4head_classifier.model', 'wb'))

## Trenowanie
3 klasy: konw, niekonw, inne pozycje

In [ ]:
### sprawdzanie ze introny nie sa swoimi wlasnymi wariantami

seqs = [i for i in SeqIO.parse("./fastas_and_gff/introns_for_3-class_classifier_training.fasta", 'fasta')]
for id1 in range(len(seqs)-1):
    s1 = seqs[id1]
    for id2 in range(id1+1, len(seqs)):
        s2 = seqs[id2]
        if s1.seq==s2.seq:
            print(s1.description)
            print(s2.description)
            print('\n')

In [ ]:
introns_seqs, introns_classes, introns_characteristics = [], [], []
f = open("./fastas_and_gff/introns_for_3-class_classifier_training.fasta", 'r')
for l in f.readlines():
    if l.startswith(">"):
        t = l.split()[-1]
        intron_type = 0 if t=="intron_K" else (1 if t=="intron_NK" else 2)
    else:
        prev_exon_seq, intron_seq, next_exon_seq = l[:5], l[5:-5], l[-5:]
        its_characteristics = introns.compute_intron_characteristics(l)
        introns_seqs.append((prev_exon_seq, intron_seq, next_exon_seq))
        introns_classes.append(intron_type)
        introns_characteristics.append(its_characteristics)
print(len(introns_seqs), len(introns_classes), len(introns_characteristics))

introns_classes = np.array(introns_classes).reshape(-1,1)

X_train, X_test, y_train, y_test = train_test_split(introns_characteristics, introns_classes, test_size=0.3, random_state=420)
classes_weights_3head = class_weight.compute_sample_weight(class_weight='balanced', y=y_train)

In [ ]:
#sprawdzanie korelacji miedzy cechami
pd.DataFrame(introns_characteristics).corr()

In [ ]:
# clf_3head = MultiOutputClassifier(xgb.XGBClassifier(use_label_encoder=False,
#                                                     eval_metric='aucpr',
#                                                     objective='multi:softmax'))
clf_3head = MultiOutputClassifier(xgb.XGBClassifier(use_label_encoder=False,
                                                    eval_metric=balanced_accuracy_score,
                                                    objective='multi:softmax',
                                                    seed=123))
clf_3head.fit(X_train, y_train, sample_weight=classes_weights_3head);

pred_3head = clf_3head.predict(X_test)

In [ ]:
acc = balanced_accuracy_score(y_test,pred_3head)
print("3-class classifier accuracy score:", acc)

precision, recall, fscore, _ = precision_recall_fscore_support(y_test,pred_3head)
#print("Precision:\t", precision)
#print("recall:\t\t", recall)
#print("f-score:\t", fscore)
print(f'''
Precision:    {precision}
recall:       {recall}
f-score:      {fscore}
''')
classifiers_acc_prec_rec_f['multi 3-class'] = (acc, precision, recall, fscore)

print(multilabel_confusion_matrix(y_test,pred_3head))

In [ ]:
### pickle.dump(clf_3head, open('3head_classifier.model', 'wb'))

## Trenowanie
4 klasy: konw, niekonw, intermediate, losowe sekwencje

In [ ]:
introns_seqs, introns_classes, introns_characteristics = [], [], []
#f = open("introns_for_classifier_training_w_random.fasta", 'r')
f = open("./fastas_and_gff/introns_for_3-class_classifier_training.fasta", 'r')
for l in f.readlines():
    if l.startswith(">"):
        t = l.split()[-1]
        intron_type = 0 if t=="intron_K" else (1 if t=="intron_NK" else (2 if t=="intron_I" else 3))
    else:
        #prev_exon_seq, intron_seq, next_exon_seq = l[:5], l[5:-5], l[-5:]
        #its_characteristics = introns.compute_intron_characteristics(prev_exon_seq, intron_seq, next_exon_seq)
        its_characteristics = introns.compute_intron_characteristics(l)
        introns_seqs.append((prev_exon_seq, intron_seq, next_exon_seq))
        introns_classes.append(intron_type)
        introns_characteristics.append(its_characteristics)
print(len(introns_seqs), len(introns_classes), len(introns_characteristics))

introns_classes = np.array(introns_classes).reshape(-1,1)

X_train, X_test, y_train, y_test = train_test_split(introns_characteristics, introns_classes, test_size=0.3, random_state=420)


clf_4head = MultiOutputClassifier(xgb.XGBClassifier(eval_metric='aucpr'))
clf_4head.fit(X_train, y_train);

pred_4head = clf_4head.predict(X_test)


acc = accuracy_score(y_test,pred_4head)
print("4-class classifier accuracy score:", acc)

precision, recall, fscore, _ = precision_recall_fscore_support(y_test,pred_4head)
print(f'''
Precision:    {precision}
recall:       {recall}
f-score:      {fscore}
''')
classifiers_acc_prec_rec_f['multi 4-class'] = (acc, precision, recall, fscore)

print(multilabel_confusion_matrix(y_test,pred_4head))

## Trenowanie - multiklasy

In [ ]:
introns_seqs, introns_classes, introns_characteristics = [], [], []
f = open("./fastas_and_gff/introns_for_3-class_classifier_training.fasta", 'r')
for l in f.readlines():
    if l.startswith(">"):
        t = l.split()[-1]
        intron_type = 0 if t=="intron_K" else (1 if t=="intron_NK" else 2)
    else:
        prev_exon_seq, intron_seq, next_exon_seq = l[:5], l[5:-5], l[-5:]
        its_characteristics = introns.compute_intron_characteristics(prev_exon_seq+intron_seq+next_exon_seq)
        introns_seqs.append((prev_exon_seq, intron_seq, next_exon_seq))
        introns_classes.append(intron_type)
        introns_characteristics.append(its_characteristics)
print(len(introns_seqs), len(introns_classes), len(introns_characteristics))

introns_classes_multi = []
for i in introns_classes:
    is_k = True if i==0 else False
    is_nk = True if i==1 else False
    introns_classes_multi.append([is_k, is_nk])
    
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(introns_characteristics, introns_classes_multi, test_size=0.3, random_state=420)
classes_weights_multi = class_weight.compute_sample_weight(class_weight='balanced', y=y_train)

In [ ]:
#clf_multi = MultiOutputClassifier(xgb.XGBClassifier(eval_metric='aucpr'))
clf_multi = MultiOutputClassifier(xgb.XGBClassifier(eval_metric=balanced_accuracy_score, seed=123))
clf_multi.fit(X_train_multi, y_train_multi, sample_weight=classes_weights_multi);

In [ ]:
pred_multi = clf_multi.predict(X_test_multi)


#print("Multiclass classifier accuracy score:", balanced_accuracy_score(y_test_multi,pred_multi))

precision, recall, fscore, _ = precision_recall_fscore_support(y_test_multi,pred_multi)
print(f'''
Precision:    {precision}
recall:       {recall}
f-score:      {fscore}
''')
print(multilabel_confusion_matrix(y_test_multi,pred_multi))

classifiers_acc_prec_rec_f['double multilabel'] = (0, precision, recall, fscore)

In [ ]:
### pickle.dump(clf_multi, open('multi_classifier.model', 'wb'))

Comparison of classifiers

In [ ]:
classifiers = classifiers_acc_prec_rec_f.keys()
classifiers = [c.replace(" ", "\n") for c in classifiers]
accuracies = [x[0] for x in classifiers_acc_prec_rec_f.values()]
precisions = [x[1] for x in classifiers_acc_prec_rec_f.values()]
recalls = [x[2] for x in classifiers_acc_prec_rec_f.values()]
fscores = [x[3] for x in classifiers_acc_prec_rec_f.values()]

fig, ax = plt.subplots(1,4,figsize=(20,4))
ax = ax.flatten()

#for classifier, a in zip(classifiers,accuracies):
ax[0].bar(classifiers, accuracies, color=teal_color)
ax[0].set_title('Accuracy')
ax[0].set_ylim((0,1))

ax[1].bar(classifiers, [np.mean(l) for l in precisions], color=teal_color)
ax[1].set_title('Precision')
ax[1].set_ylim((0,1))

ax[2].bar(classifiers, [np.mean(l) for l in recalls], color=teal_color)
ax[2].set_title('Recall')
ax[2].set_ylim((0,1))

ax[3].bar(classifiers, [np.mean(l) for l in fscores], color=teal_color)
ax[3].set_title('F-score')
ax[3].set_ylim((0,1))

plt.savefig('3_classifiers_scores.png')
plt.show()

## Stats w/ classifiers

In [ ]:
clf_3head = pickle.load(open('./Models/3head_classifier.model', 'rb'))
#clf_4head = pickle.load(open('4head_classifier.model', 'rb'))
#clf_multi = pickle.load(open('multi_classifier.model', 'rb'))